# ExplicitStimulusWhiskerData

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/ExplicitStimulusWhiskerData.md`


Notebook source link: [ExplicitStimulusWhiskerData.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/ExplicitStimulusWhiskerData.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "ExplicitStimulusWhiskerData"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"ExplicitStimulusWhiskerData: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"ExplicitStimulusWhiskerData: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"ExplicitStimulusWhiskerData: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"ExplicitStimulusWhiskerData: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "[~,~,explicitStimulusDir] = getPaperDataDirs();",
    "Direction=3; Neuron=1; Stim=2;",
    "datapath = fullfile(explicitStimulusDir,strcat('Dir', num2str(Direction)),...",
    "strcat('Neuron', num2str(Neuron)), strcat('Stim', num2str(Stim)));",
    "data=load(fullfile(datapath,'trngdataBis.mat'));",
    "time=0:.001:(length(data.t)-1)*.001;",
    "stimData = data.t;",
    "spikeTimes = time(data.y==1);",
    "stim = Covariate(time,stimData,'Stimulus','time','s','V',{'stim'});",
    "baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...",
    "{'constant'});",
    "nst = nspikeTrain(spikeTimes);",
    "nspikeColl = nstColl(nst);",
    "cc = CovColl({stim,baseline});",
    "trial = Trial(nspikeColl,cc);",
    "trial.plot;",
    "figure;",
    "subplot(2,1,1);",
    "nst2 = nspikeTrain(spikeTimes);",
    "nst2.setMaxTime(21);nst.plot;",
    "subplot(2,1,2);",
    "stim.getSigInTimeWindow(0,21).plot;",
    "clear c;",
    "selfHist = [] ; NeighborHist = []; sampleRate = 1000;",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,NeighborHist);",
    "c{1}.setName('Baseline');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "figure;",
    "results.Residual.xcov(stim).windowedSignal([0,1]).plot;",
    "[m,ind,ShiftTime] = max(results.Residual.xcov(stim).windowedSignal([0,1]));",
    "stim = Covariate(time,stimData,'Stimulus','time','s','V',{'stim'});",
    "stim = stim.shift(ShiftTime);",
    "baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...",
    "{'constant'});",
    "nst = nspikeTrain(spikeTimes);",
    "nspikeColl = nstColl(nst);",
    "cc = CovColl({stim,baseline});",
    "trial = Trial(nspikeColl,cc);",
    "clear c;",
    "selfHist = [] ; NeighborHist = []; sampleRate = 1000;",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,...",
    "NeighborHist);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,selfHist,NeighborHist);",
    "c{2}.setName('Baseline+Stimulus');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results.plotResults;",
    "sampleRate=1000;",
    "delta=1/sampleRate*1;",
    "maxWindow=1; numWindows=30;",
    "windowTimes =unique(round([0 logspace(log10(delta),...",
    "log10(maxWindow),numWindows)]*sampleRate)./sampleRate);",
    "results =Analysis.computeHistLagForAll(trial,windowTimes,...",
    "{{'Baseline','constant'},{'Stimulus','stim'}},'BNLRCG',0,sampleRate,0);",
    "KSind = find(results{1}.KSStats.ks_stat == min(results{1}.KSStats.ks_stat));",
    "AICind = find((results{1}.AIC(2:end)-results{1}.AIC(1))== ...",
    "min(results{1}.AIC(2:end)-results{1}.AIC(1)));",
    "BICind = find((results{1}.BIC(2:end)-results{1}.BIC(1))== ...",
    "min(results{1}.BIC(2:end)-results{1}.BIC(1)));",
    "if(AICind==1)",
    "AICind=inf;",
    "end",
    "if(BICind==1)",
    "BICind=inf; %sometime BIC is non-decreasing and the index would be 1",
    "end",
    "windowIndex = min([KSind,AICind,BICind]) %use the minimum order model",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;",
    "clear c;",
    "if(windowIndex>1)",
    "selfHist = windowTimes(1:windowIndex);",
    "else",
    "selfHist = [];",
    "end",
    "NeighborHist = []; sampleRate = 1000;",
    "figure;",
    "x=1:length(windowTimes);",
    "subplot(3,1,1); plot(x,results{1}.KSStats.ks_stat,'.'); axis tight; hold on;",
    "plot(x(windowIndex),results{1}.KSStats.ks_stat(windowIndex),'r*');",
    "set(gca,'xtick',[]);",
    "ylabel('KS Statistic');",
    "dAIC = results{1}.AIC-results{1}.AIC(1);",
    "subplot(3,1,2); plot(x,dAIC,'.');",
    "set(gca,'xtick',[]);",
    "ylabel('\\Delta AIC');axis tight; hold on;",
    "plot(x(windowIndex),dAIC(windowIndex),'r*');",
    "dBIC = results{1}.BIC-results{1}.BIC(1);",
    "subplot(3,1,3); plot(x,dBIC,'.');",
    "ylabel('\\Delta BIC'); axis tight; hold on;",
    "plot(x(windowIndex),dBIC(windowIndex),'r*');",
    "for i=2:length(x)",
    "histLabels{i} = ['[' num2str(windowTimes(i-1),3) ',' num2str(windowTimes(i),3) ,']'];",
    "end",
    "figure;",
    "plot(x,dBIC,'.');",
    "xticks = 1:(length(histLabels));",
    "set(gca,'xtick',xticks,'xtickLabel',histLabels,'FontSize',6);",
    "if(max(xticks)>=1)",
    "xticklabel_rotate([],90,[],'Fontsize',8);",
    "end",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,[],NeighborHist);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,[],[]);",
    "c{2}.setName('Baseline+Stimulus');",
    "c{3} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,windowTimes(1:windowIndex),[]);",
    "c{3}.setName('Baseline+Stimulus+Hist');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results.plotResults;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for ExplicitStimulusWhiskerData.")


In [ ]:
# ExplicitStimulusWhiskerData: stimulus-locked spiking with binomial GLM fit.
from pathlib import Path
import nstat
from scipy.io import loadmat
fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/ExplicitStimulusWhiskerData_gold.mat"
m = loadmat(str(fixture_path))
time = np.asarray(m["time_ws"], dtype=float).reshape(-1); stimulus = np.asarray(m["stimulus_ws"], dtype=float).reshape(-1); spike = np.asarray(m["spike_ws"], dtype=float).reshape(-1)
expected_prob = np.asarray(m["expected_prob_ws"], dtype=float).reshape(-1); expected_rmse = float(np.asarray(m["expected_rmse_ws"], dtype=float).reshape(-1)[0])
fit = Analysis.fit_glm(X=stimulus[:, None], y=spike, fit_type="binomial", dt=1.0); pred_prob = np.asarray(fit.predict(stimulus[:, None]), dtype=float).reshape(-1)
window = np.ones(25, dtype=float) / 25.0; spike_prob = np.convolve(spike, window, mode="same")

fig, axes = plt.subplots(3, 1, figsize=(9.5, 7.2), sharex=False)
axes[0].plot(time, stimulus, color="k", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: explicit stimulus")
axes[0].set_ylabel("z-score")

axes[1].vlines(time[spike > 0.0], 0.6, 1.4, linewidth=0.4)
axes[1].set_ylabel("trial #1")
axes[1].set_title("Spike raster (MATLAB fixture trial)")

axes[2].plot(time, spike_prob, color="tab:blue", linewidth=1.0, label="smoothed observed")
axes[2].plot(time, pred_prob, color="tab:red", linewidth=1.0, label="python fit")
axes[2].plot(time, expected_prob, color="tab:green", linewidth=0.9, linestyle="--", label="matlab gold")
axes[2].set_title("Observed and fitted spike probability")
axes[2].set_xlabel("time [s]")
axes[2].set_ylabel("p(spike)")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

fit_rmse = float(np.sqrt(np.mean((pred_prob - spike) ** 2))); prob_max_abs = float(np.max(np.abs(pred_prob - expected_prob)))
assert pred_prob.shape == expected_prob.shape
assert prob_max_abs < 0.1
assert abs(fit_rmse - expected_rmse) < 0.1
CHECKPOINT_METRICS = {
    "prob_max_abs": float(prob_max_abs),
    "fit_rmse": float(fit_rmse),
}
CHECKPOINT_LIMITS = {
    "prob_max_abs": (0.0, 0.1),
    "fit_rmse": (0.0, 0.5),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
